# Create Training Labels from the Critical Mineral Deposits Database
from shapefile to GeoTIFF


In [1]:
import os
import numpy as np
import geopandas as gpd

import sys
if sys.version_info < (3, 9):
    from importlib_resources import files
else:
    from importlib.resources import files

from beak.utilities.io import load_raster
from beak.utilities.conversions import create_binary_raster


# Load data

**User definitions**

In [2]:
# Set base paths and files
BASE_PATH = files("beak.data")

EPSG_CODE = 102008
RESOLUTION = 2240
BASE_SPATIAL = str(EPSG_CODE) + "_" + str(RESOLUTION)
BASE_EXTENT = "us_cont"
BASE_RASTER = BASE_PATH / "BASE_RASTERS" / str("EPSG_" + str(EPSG_CODE) + "_RES_" + str(RESOLUTION) + "_" + BASE_EXTENT + ".tif")

# Points file and query to select relevant mineral occurences
PATH_SHAPEFILE = BASE_PATH / "RAW" / "mineral_deposits" / "Critical_Mineral_Deposits" / "critical_mineral_deposits.shp"
SQL_QUERY = "DepType == 'Nickel-copper-PGE sulfide'"

# Set the output file
PATH_ROOT = BASE_PATH / "PROCESSED" / str("national" + "_" + BASE_EXTENT + "_" + BASE_SPATIAL)
PATH_EXPORT = PATH_ROOT / "labels" / str(BASE_RASTER.stem + "_MAMA_NICO_HM6_CMD" + ".tif")
OUT_FILE = PATH_EXPORT

print(f"Output file: {OUT_FILE}")


Output file: s:\projekte\20230082_darpa_criticalmaas_ta3\bearbeitung\github\beak-ta3\src\beak\data\PROCESSED\national_102008_2240_us_cont\labels\EPSG_102008_RES_2240_us_cont_MAMA_NICO_CMD_NCPGE_SULFIDE.tif


# Create Labels


Reproject to **BASE RASTER** CRS

If the reprojection does not work (geometry points are **inf**), run the cell a second time.

In [4]:
from beak.utilities.vector_processing import _reproject_vector_data

base_raster = load_raster(BASE_RASTER)
template_crs = base_raster.crs
gdf = _reproject_vector_data(PATH_SHAPEFILE, crs=template_crs, encoding="utf-8", query=SQL_QUERY)
gdf.head()

,Deposit,Lat_WGS84,Long_WGS84,State,MinSystem,DepType,CritMin,FocusArea,DepCat,Rank,Prduction1,Prduction2,Resources1,Resources2,Resources3,Source_s,Links,geometry
45,Benbow nickel-copper deposit,45.36094,-109.81588,MT,Mafic magmatic,Nickel-copper-PGE sulfide,nickel,Stillwater Complex,CM resources,3,None.,None,"Resource (1993): 13,100,000 metric tons ore @0...",None,None,Wetzel (1986); Suda and others (2009),None,POINT (-1014608.287 707415.104)
54,Birch Lake,47.73294,-91.80817,MN,Mafic magmatic,Nickel-copper-PGE sulfide,"nickel, PGE",Midcontinent Rift large mafic intrusions,CM resources,3,None.,None,Birch Lake resource (2014): Inferred mineral r...,0.235 ppm Pt.,None,Barber and others (2014); Burger and others (2...,https://doi.org/10.5066/P9V74HIU,POINT (296453.944 919898.314)
78,Bohemia Basin,57.97600,-136.42250,AK,Mafic magmatic,Nickel-copper-PGE sulfide,"cobalt, nickel",Southeast Alaska Fairweather trend PGE,CM resources,3,None.,None,The Basin and Takanis deposits have proven res...,None,None,ARDF (1996),https://mrdata.usgs.gov/ardf/show-ardf.php?ard...,POINT (-2281900.306 2596700.488)
93,Brady Glacier,58.55235,-136.93204,AK,Mafic magmatic,Nickel-copper-PGE sulfide,"cobalt, nickel",Southeast Alaska Fairweather trend PGE,CM resources,3,None.,None,"Proven reserve (1995): 91,000,000 tonnes ore @...",None,None,Burger and others (2018),https://doi.org/10.5066/P9V74HIU,POINT (-2281611.200 2668189.264)
145,Chrome Mountain Hybrid and Dunite Ridge nickel...,45.43788,-110.12507,MT,Mafic magmatic,Nickel-copper-PGE sulfide,"cobalt, nickel, PGE",Stillwater Complex,CM resources,3,None.,None,Inferred in-pit mineral resource estimates (20...,None,None,Childs and Armitage (2021),None,POINT (-1035798.613 719750.894)


Create binary label raster

In [5]:
out_array = create_binary_raster(gdf, base_raster, query=SQL_QUERY, all_touched=False, same_shape=True, fill_negatives=True, out_file=OUT_FILE)
print(f"Number of positive training labels: {np.sum(out_array==1)}")

Number of positive training labels: 19
